# 00 Data Preparation

This notebook prepares the breast cancer DepMap/CCLE + PRISM doxorubicin modelling table.

The goal is to join baseline molecular features from DepMap/CCLE with measured PRISM doxorubicin response for the same breast cancer cell lines.

## Why This Table Is Needed

DepMap/CCLE provides baseline molecular measurements for untreated cancer cell lines. In this notebook we focus on gene expression for a small p53 and DNA-damage response gene set.

PRISM provides drug-response measurements for cancer cell lines. Here we use PRISM doxorubicin log-fold-change as the response outcome.

The joined table will later support:

- **Q2**: testing whether p53 or ODE-derived features predict doxorubicin response in breast cancer cell lines;
- **Q3**: deriving a cell-line doxorubicin-response signature and applying it to TCGA-BRCA patients;
- **Q5**: comparing mechanistic p53 features with regression or machine-learning approaches.

## Important Conceptual Point

PRISM log-fold-change is **not gene-expression change**.

In this workflow:

- DepMap/CCLE expression is the predictor matrix `X`.
- PRISM doxorubicin log-fold-change is the outcome `y`.

More negative PRISM log-fold-change generally means lower relative cell abundance or viability after doxorubicin treatment compared with control. That is interpreted as greater sensitivity.

The PRISM LFC should not be described as "percentage of cells destroyed". It is better described as relative cell abundance or viability versus control.

## Expected Raw Files

Place the DepMap/CCLE and PRISM files here:

```text
data/raw/depmap_data/
```

Expected files:

- `Model.csv`
- `OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv`
- `primary-screen-replicate-collapsed-treatment-info.csv`
- `primary-screen-replicate-collapsed-logfold-change.csv`

If these files are missing, this notebook stops after the availability check and does not fabricate a processed table.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

raw_dir = project_root / "data/raw/depmap_data"
processed_dir = project_root / "data/processed"
figures_dir = project_root / "figures"
tables_dir = project_root / "tables"

required_files = {
    "model_metadata": raw_dir / "Model.csv",
    "expression": raw_dir / "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv",
    "prism_treatment_info": raw_dir / "primary-screen-replicate-collapsed-treatment-info.csv",
    "prism_lfc": raw_dir / "primary-screen-replicate-collapsed-logfold-change.csv",
}

file_check = pd.DataFrame(
    {
        "input": required_files.keys(),
        "path": [str(path) for path in required_files.values()],
        "available": [path.exists() for path in required_files.values()],
    }
)

missing_files = file_check.loc[~file_check["available"], "path"].tolist()
raw_files_available = len(missing_files) == 0

display(file_check)

if missing_files:
    print("Missing raw files. Place these files under data/raw/depmap_data/:")
    for path in missing_files:
        print(f"- {path}")

## Load DepMap/CCLE Model Metadata

First we inspect `Model.csv`, identify useful columns, and filter to breast cancer cell lines.

The most useful fields are expected to include a DepMap model ID, a cell-line name, lineage, and cancer type or primary disease.

In [ ]:
if raw_files_available:
    model_metadata = pd.read_csv(required_files["model_metadata"])

    print("Model.csv shape:", model_metadata.shape)
    print("Model.csv columns:")
    print(list(model_metadata.columns))

    id_col = "ModelID"
    name_col = "StrippedCellLineName" if "StrippedCellLineName" in model_metadata.columns else "ModelID"
    lineage_col = "OncotreeLineage" if "OncotreeLineage" in model_metadata.columns else None
    cancer_type_col = "OncotreePrimaryDisease" if "OncotreePrimaryDisease" in model_metadata.columns else None

    print("Selected ID column:", id_col)
    print("Selected cell-line name column:", name_col)
    print("Selected lineage column:", lineage_col)
    print("Selected cancer-type column:", cancer_type_col)

    lineage_text = model_metadata[lineage_col].fillna("") if lineage_col else pd.Series("", index=model_metadata.index)
    cancer_type_text = model_metadata[cancer_type_col].fillna("") if cancer_type_col else pd.Series("", index=model_metadata.index)

    breast_mask = lineage_text.str.contains("breast", case=False, na=False) | cancer_type_text.str.contains("breast", case=False, na=False)
    breast_models = model_metadata.loc[breast_mask].copy()

    breast_model_cols = [id_col, name_col]
    if lineage_col:
        breast_model_cols.append(lineage_col)
    if cancer_type_col:
        breast_model_cols.append(cancer_type_col)

    breast_models = breast_models[breast_model_cols].drop_duplicates()
    breast_models = breast_models.rename(columns={id_col: "depmap_id", name_col: "cell_line_name"})

    display(breast_models.head())
else:
    print("Skipping metadata loading until the raw files are available.")

## Load DepMap/CCLE Expression

This section keeps a small p53 and DNA-damage response gene set if those genes are present in the expression matrix.

The target gene set is:

`ATM`, `CHEK2`, `HIPK2`, `MDM2`, `PPM1D`, `SIAH1`, `TP53`, `WSB1`, `CDKN1A`, `BAX`, `BBC3`, `GADD45A`, `MDM4`, `ATR`, `CHEK1`, `CASP3`.

In [ ]:
p53_ddr_genes = [
    "ATM", "CHEK2", "HIPK2", "MDM2", "PPM1D", "SIAH1", "TP53", "WSB1",
    "CDKN1A", "BAX", "BBC3", "GADD45A", "MDM4", "ATR", "CHEK1", "CASP3",
]

if raw_files_available:
    expression = pd.read_csv(required_files["expression"])
    print("Expression matrix shape:", expression.shape)
    print("First expression columns:")
    print(list(expression.columns[:10]))

    expression_id_col = "ModelID" if "ModelID" in expression.columns else expression.columns[0]
    print("Selected expression ID column:", expression_id_col)

    gene_name_map = {}
    for column in expression.columns:
        clean_gene = column.split(" (")[0]
        if clean_gene in p53_ddr_genes and clean_gene not in gene_name_map:
            gene_name_map[clean_gene] = column

    found_genes = [gene for gene in p53_ddr_genes if gene in gene_name_map]
    missing_genes = [gene for gene in p53_ddr_genes if gene not in gene_name_map]

    print("Found genes:", found_genes)
    print("Missing genes:", missing_genes)

    expression_keep_cols = [expression_id_col] + [gene_name_map[gene] for gene in found_genes]
    expression_small = expression[expression_keep_cols].copy()
    expression_small = expression_small.rename(columns={expression_id_col: "depmap_id"})
    expression_small = expression_small.rename(columns={gene_name_map[gene]: gene for gene in found_genes})

    display(expression_small.head())
else:
    print("Skipping expression loading until the raw files are available.")

## Load PRISM Treatment Information

This section searches the PRISM treatment metadata for doxorubicin. If multiple doxorubicin treatment IDs are present, we keep all of them initially and aggregate by median response per cell line later.

In [ ]:
if raw_files_available:
    treatment_info = pd.read_csv(required_files["prism_treatment_info"])

    print("Treatment info shape:", treatment_info.shape)
    print("Treatment info columns:")
    print(list(treatment_info.columns))

    treatment_name_col = "name" if "name" in treatment_info.columns else treatment_info.columns[0]
    treatment_id_col = "column_name" if "column_name" in treatment_info.columns else treatment_info.columns[0]

    doxorubicin_rows = treatment_info[
        treatment_info[treatment_name_col].astype(str).str.contains("doxorubicin", case=False, na=False)
    ].copy()

    doxorubicin_treatment_ids = doxorubicin_rows[treatment_id_col].dropna().astype(str).tolist()

    print("Selected treatment name column:", treatment_name_col)
    print("Selected treatment ID column:", treatment_id_col)
    print("Number of matched doxorubicin treatments:", len(doxorubicin_treatment_ids))
    display(doxorubicin_rows)
else:
    print("Skipping PRISM treatment-info loading until the raw files are available.")

## Load PRISM Doxorubicin Response

This section extracts the PRISM doxorubicin log-fold-change values and aggregates multiple doxorubicin treatment columns by median response per cell line.

The original PRISM doxorubicin response is kept as `prism_doxorubicin_lfc`.

In [ ]:
if raw_files_available:
    prism_lfc = pd.read_csv(required_files["prism_lfc"])

    print("PRISM LFC matrix shape:", prism_lfc.shape)
    print("First PRISM LFC columns:")
    print(list(prism_lfc.columns[:10]))

    prism_id_col = "ModelID" if "ModelID" in prism_lfc.columns else prism_lfc.columns[0]
    available_dox_cols = [col for col in doxorubicin_treatment_ids if col in prism_lfc.columns]

    print("Selected PRISM cell-line ID column:", prism_id_col)
    print("Doxorubicin response columns found in LFC matrix:", available_dox_cols)

    if available_dox_cols:
        prism_dox = prism_lfc[[prism_id_col] + available_dox_cols].copy()
        prism_dox["prism_doxorubicin_lfc"] = prism_dox[available_dox_cols].median(axis=1, skipna=True)
        prism_dox = prism_dox.rename(columns={prism_id_col: "depmap_id"})
        prism_dox = prism_dox[["depmap_id", "prism_doxorubicin_lfc"]].dropna()
        display(prism_dox.head())
    else:
        prism_dox = pd.DataFrame(columns=["depmap_id", "prism_doxorubicin_lfc"])
        print("No doxorubicin response columns were found in the PRISM LFC matrix.")
else:
    print("Skipping PRISM response loading until the raw files are available.")

## Join Breast Cancer Metadata, Expression, and PRISM Response

The modelling table joins:

```text
baseline DepMap/CCLE expression -> PRISM doxorubicin response
```

This is the supervised learning table for later modelling.

In [ ]:
if raw_files_available:
    breast_with_expression = breast_models.merge(expression_small, on="depmap_id", how="inner")
    breast_with_prism = breast_models.merge(prism_dox, on="depmap_id", how="inner")

    modelling_table = breast_with_expression.merge(prism_dox, on="depmap_id", how="inner")

    modelling_table["relative_abundance_vs_control"] = 2 ** modelling_table["prism_doxorubicin_lfc"]
    modelling_table["relative_abundance_percent"] = 100 * modelling_table["relative_abundance_vs_control"]
    modelling_table["doxorubicin_sensitivity_score"] = -1 * modelling_table["prism_doxorubicin_lfc"]

    metadata_cols = ["depmap_id", "cell_line_name"]
    if lineage_col:
        metadata_cols.append(lineage_col)
    if cancer_type_col:
        metadata_cols.append(cancer_type_col)

    response_cols = [
        "prism_doxorubicin_lfc",
        "relative_abundance_vs_control",
        "relative_abundance_percent",
        "doxorubicin_sensitivity_score",
    ]
    gene_cols = [gene for gene in found_genes if gene in modelling_table.columns]

    modelling_table = modelling_table[metadata_cols + response_cols + gene_cols].copy()

    display(modelling_table.head())
else:
    print("Skipping the final join until the raw files are available.")

## Dataset Summary and Outputs

When the raw files are available, this section saves:

- `data/processed/brca_prism_doxorubicin_modelling_table.csv`
- `tables/brca_prism_dataset_summary.csv`
- `figures/prism_doxorubicin_lfc_distribution.png`

In [ ]:
if raw_files_available:
    processed_dir.mkdir(exist_ok=True)
    tables_dir.mkdir(exist_ok=True)
    figures_dir.mkdir(exist_ok=True)

    summary = pd.DataFrame(
        {
            "metric": [
                "total DepMap models",
                "breast cancer models",
                "breast cancer models with expression data",
                "breast cancer models with PRISM doxorubicin response",
                "final joined modelling table rows",
            ],
            "value": [
                len(model_metadata),
                len(breast_models),
                len(breast_with_expression),
                len(breast_with_prism),
                len(modelling_table),
            ],
        }
    )

    modelling_table_path = processed_dir / "brca_prism_doxorubicin_modelling_table.csv"
    summary_path = tables_dir / "brca_prism_dataset_summary.csv"
    figure_path = figures_dir / "prism_doxorubicin_lfc_distribution.png"

    modelling_table.to_csv(modelling_table_path, index=False)
    summary.to_csv(summary_path, index=False)

    plt.figure(figsize=(7, 4.5))
    plt.hist(modelling_table["prism_doxorubicin_lfc"].dropna(), bins=15, edgecolor="black")
    plt.axvline(0, color="black", linestyle="--", linewidth=1)
    plt.xlabel("PRISM doxorubicin log-fold-change")
    plt.ylabel("Number of breast cancer cell lines")
    plt.title("Distribution of PRISM doxorubicin response")
    plt.tight_layout()
    plt.savefig(figure_path, dpi=200)
    plt.show()

    display(summary)
    print("Saved modelling table to:", modelling_table_path)
    print("Saved summary table to:", summary_path)
    print("Saved figure to:", figure_path)
else:
    print("No processed table, summary CSV, or figure was created because raw files are missing.")

## Interpretation

When the raw files are available and the notebook is run, report:

- how many DepMap models were available;
- how many were breast cancer cell lines;
- how many breast cancer cell lines had expression data;
- how many breast cancer cell lines had PRISM doxorubicin response;
- how many rows are in the final joined modelling table.

A negative PRISM doxorubicin LFC means lower relative abundance or viability compared with control after doxorubicin treatment. In this notebook, `doxorubicin_sensitivity_score = -1 * prism_doxorubicin_lfc`, so higher sensitivity scores mean greater apparent sensitivity.

This table is the basis for later regression and signature modelling because it links baseline cell-line expression features to measured doxorubicin response. No lasso, elastic net, survival analysis, GDSC analysis, LINCS analysis, or p53 ODE implementation is run here.

## PRISM/DepMap Cell-Line Attrition

This section documents how the final PRISM/DepMap doxorubicin modelling cohort is formed.

The final `n = 26` is **not** the total number of breast cancer models in DepMap. It is the intersection of:

- models labelled as breast lineage or breast primary disease;
- models with DepMap/CCLE expression data;
- models with the selected p53/DNA-damage expression columns;
- PRISM treatment metadata containing doxorubicin;
- PRISM log-fold-change response data for the matched doxorubicin treatment columns.

This small sample size is important for Q3 and Q4 interpretation. It helps explain why Q3 cross-validation was unstable and why transfer analyses based on PRISM response should be described as exploratory.


In [ ]:
attrition_table_path = tables_dir / "prism_depmap_attrition_table.csv"
attrition_figure_path = figures_dir / "prism_depmap_attrition_funnel.png"
attrition_note_path = project_root / "notes/prism_depmap_attrition_summary.md"

for folder in [attrition_table_path.parent, attrition_figure_path.parent, attrition_note_path.parent]:
    folder.mkdir(exist_ok=True)

attrition_rows = []

if raw_files_available:
    total_depmap_models = model_metadata[id_col].dropna().nunique()
    breast_model_count = breast_models["depmap_id"].dropna().nunique()
    expression_model_count = expression_small["depmap_id"].dropna().nunique()
    breast_expression_count = breast_with_expression["depmap_id"].dropna().nunique()

    breast_with_selected_gene_expression = breast_with_expression.dropna(subset=found_genes).copy()
    breast_selected_gene_count = breast_with_selected_gene_expression["depmap_id"].dropna().nunique()

    doxorubicin_treatment_row_count = len(doxorubicin_rows)

    prism_lfc_model_ids = prism_lfc[prism_id_col].dropna().astype(str).unique()
    breast_any_prism_count = breast_models[breast_models["depmap_id"].astype(str).isin(prism_lfc_model_ids)]["depmap_id"].nunique()

    breast_non_missing_dox_count = breast_with_prism["depmap_id"].dropna().nunique()
    final_joined_count = modelling_table["depmap_id"].dropna().nunique()

    attrition_rows = [
        {
            "step_order": 1,
            "attrition_step": "Total rows/models in DepMap Model.csv",
            "count": total_depmap_models,
            "count_type": "unique DepMap model IDs",
            "note": f"ID column: {id_col}",
        },
        {
            "step_order": 2,
            "attrition_step": "Models labelled as breast cancer / breast lineage",
            "count": breast_model_count,
            "count_type": "unique DepMap model IDs",
            "note": f"Filtered using {lineage_col} and {cancer_type_col}",
        },
        {
            "step_order": 3,
            "attrition_step": "Breast cancer models with expression data",
            "count": breast_expression_count,
            "count_type": "unique DepMap model IDs",
            "note": f"Expression table contains {expression_model_count} total model IDs",
        },
        {
            "step_order": 4,
            "attrition_step": "Breast cancer models with selected p53/DNA-damage gene expression columns",
            "count": breast_selected_gene_count,
            "count_type": "unique DepMap model IDs",
            "note": f"Genes found: {len(found_genes)} of {len(p53_ddr_genes)}",
        },
        {
            "step_order": 5,
            "attrition_step": "Doxorubicin treatment rows found in PRISM treatment info",
            "count": doxorubicin_treatment_row_count,
            "count_type": "treatment metadata rows",
            "note": f"Matched by searching {treatment_name_col} for doxorubicin",
        },
        {
            "step_order": 6,
            "attrition_step": "Breast cancer models with any PRISM response data",
            "count": breast_any_prism_count,
            "count_type": "unique DepMap model IDs",
            "note": f"Any row in {required_files['prism_lfc'].name}",
        },
        {
            "step_order": 7,
            "attrition_step": "Breast cancer models with non-missing PRISM doxorubicin LFC",
            "count": breast_non_missing_dox_count,
            "count_type": "unique DepMap model IDs",
            "note": f"Doxorubicin LFC aggregated across {len(available_dox_cols)} available treatment column(s)",
        },
        {
            "step_order": 8,
            "attrition_step": "Final joined modelling table rows",
            "count": final_joined_count,
            "count_type": "unique DepMap model IDs / table rows",
            "note": "Intersection of breast metadata, selected expression genes, and non-missing PRISM doxorubicin response",
        },
    ]

    attrition_table = pd.DataFrame(attrition_rows)
    attrition_table.to_csv(attrition_table_path, index=False)

    display(attrition_table)
    print("Saved PRISM/DepMap attrition table to:", attrition_table_path)
else:
    attrition_table = pd.DataFrame([
        {
            "step_order": 0,
            "attrition_step": "PRISM/DepMap attrition table not generated",
            "count": 0,
            "count_type": "not available",
            "note": "Raw ignored files are required under data/raw/depmap_data/ before attrition can be reconstructed.",
        }
    ])
    attrition_table.to_csv(attrition_table_path, index=False)
    print("Raw DepMap/PRISM files are missing, so the attrition counts could not be reconstructed.")
    print("Expected raw files under data/raw/depmap_data/:")
    for path in required_files.values():
        print("-", path)


In [ ]:
if raw_files_available:
    plot_table = attrition_table[attrition_table["count_type"] != "treatment metadata rows"].copy()
    plot_table = plot_table.sort_values("step_order", ascending=True)
    plot_table["plot_label"] = [
        "All DepMap models",
        "Breast-labelled models",
        "Breast + expression",
        "Breast + 16 selected genes",
        "Breast + any PRISM response",
        "Breast + doxorubicin LFC",
        "Final modelling table",
    ]

    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    ax.barh(plot_table["plot_label"], plot_table["count"], color="#4c78a8")
    ax.invert_yaxis()
    ax.set_xlabel("Number of unique DepMap model IDs")
    ax.set_title("PRISM/DepMap doxorubicin cohort attrition")
    ax.set_xlim(0, plot_table["count"].max() * 1.18)

    for y_position, count in enumerate(plot_table["count"]):
        ax.text(count + plot_table["count"].max() * 0.015, y_position, str(count), va="center")

    plt.tight_layout()
    plt.savefig(attrition_figure_path, dpi=200)
    plt.show()

    print("Saved PRISM/DepMap attrition funnel figure to:", attrition_figure_path)
else:
    print("Attrition figure was not created because the raw DepMap/PRISM files are missing.")


In [ ]:
if raw_files_available:
    final_n = int(attrition_table.loc[attrition_table["attrition_step"] == "Final joined modelling table rows", "count"].iloc[0])
    breast_n = int(attrition_table.loc[attrition_table["attrition_step"] == "Models labelled as breast cancer / breast lineage", "count"].iloc[0])
    expression_n = int(attrition_table.loc[attrition_table["attrition_step"] == "Breast cancer models with expression data", "count"].iloc[0])
    dox_lfc_n = int(attrition_table.loc[attrition_table["attrition_step"] == "Breast cancer models with non-missing PRISM doxorubicin LFC", "count"].iloc[0])
    dox_treatment_rows = int(attrition_table.loc[attrition_table["attrition_step"] == "Doxorubicin treatment rows found in PRISM treatment info", "count"].iloc[0])

    attrition_summary_lines = [
        "# PRISM/DepMap Cell-Line Attrition Summary",
        "",
        f"Final PRISM/DepMap doxorubicin modelling cohort: {final_n} breast cancer cell lines.",
        "",
        f"This final n is not the total number of breast cancer models in DepMap. The raw metadata contain {breast_n} breast-labelled models, of which {expression_n} have the selected expression data and {dox_lfc_n} have non-missing PRISM doxorubicin log-fold-change response after matching the doxorubicin treatment metadata.",
        "",
        f"Doxorubicin treatment rows found in PRISM treatment info: {dox_treatment_rows}.",
        "",
        "The final n=26 is therefore the intersection of breast lineage, available DepMap/CCLE expression, matched PRISM doxorubicin treatment information, and non-missing PRISM doxorubicin response.",
        "",
        "This small sample size limits model generalisation and helps explain why Q3 cross-validation was unstable. It should be carried into the final report limitations section.",
    ]
else:
    attrition_summary_lines = [
        "# PRISM/DepMap Cell-Line Attrition Summary",
        "",
        "The attrition table requires the raw ignored DepMap/PRISM files under `data/raw/depmap_data/`.",
        "",
        "Expected files:",
        "",
        "- `Model.csv`",
        "- `OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv`",
        "- `primary-screen-replicate-collapsed-treatment-info.csv`",
        "- `primary-screen-replicate-collapsed-logfold-change.csv`",
    ]

attrition_note_path.write_text("\n".join(attrition_summary_lines) + "\n")
print("Saved PRISM/DepMap attrition summary to:", attrition_note_path)
print("\n".join(attrition_summary_lines))


## Interim Validation Checkpoint

This lightweight checkpoint validates the processed DepMap/CCLE + PRISM doxorubicin modelling table before any Q2, Q3, or Q5 models are built.

It does not run lasso, elastic net, random forest, Cox regression, TCGA loading, GDSC, LINCS, or the p53 ODE model.

In [ ]:
from pathlib import Path

import pandas as pd


project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

processed_table_path = project_root / "data/processed/brca_prism_doxorubicin_modelling_table.csv"
validation_summary_path = project_root / "tables/prism_data_validation_summary.csv"
validation_summary_path.parent.mkdir(exist_ok=True)

processed_table_exists = processed_table_path.exists()

print("Processed modelling table:", processed_table_path)
print("Exists:", processed_table_exists)

In [ ]:
validation_rows = []

p53_ddr_genes = [
    "ATM", "CHEK2", "HIPK2", "MDM2", "PPM1D", "SIAH1", "TP53", "WSB1",
    "CDKN1A", "BAX", "BBC3", "GADD45A", "MDM4", "ATR", "CHEK1", "CASP3",
]

if processed_table_exists:
    prism_table = pd.read_csv(processed_table_path)

    id_col = "depmap_id" if "depmap_id" in prism_table.columns else "model_id" if "model_id" in prism_table.columns else None
    cell_line_name_col = "cell_line_name" if "cell_line_name" in prism_table.columns else None
    response_cols = [col for col in prism_table.columns if "doxorubicin" in col or "relative_abundance" in col]
    present_genes = [gene for gene in p53_ddr_genes if gene in prism_table.columns]
    missing_genes = [gene for gene in p53_ddr_genes if gene not in prism_table.columns]

    print("Rows:", len(prism_table))
    print("Columns:", len(prism_table.columns))
    print("Cell-line ID column:", id_col)
    print("PRISM response columns:", response_cols)
    print("p53/DNA-damage genes present:", present_genes)
    print("p53/DNA-damage genes missing:", missing_genes)

    if cell_line_name_col:
        print("First few cell-line names:")
        display(prism_table[cell_line_name_col].head())

    display(prism_table.head())

    validation_rows.extend([
        {"check": "processed_table_exists", "status": "pass", "detail": "data/processed/brca_prism_doxorubicin_modelling_table.csv"},
        {"check": "row_count", "status": "info", "detail": len(prism_table)},
        {"check": "column_count", "status": "info", "detail": len(prism_table.columns)},
        {"check": "cell_line_id_column", "status": "pass" if id_col else "flag", "detail": id_col or "not found"},
        {"check": "p53_ddr_genes_present", "status": "info", "detail": ", ".join(present_genes)},
        {"check": "p53_ddr_genes_missing", "status": "info", "detail": ", ".join(missing_genes)},
    ])
else:
    validation_rows.append({
        "check": "processed_table_exists",
        "status": "blocker",
        "detail": "data/processed/brca_prism_doxorubicin_modelling_table.csv is not present. Run the data preparation notebook after placing the raw DepMap/PRISM files under data/raw/depmap_data/.",
    })
    print("Cannot run detailed validation because the processed modelling table is missing.")

In [ ]:
if processed_table_exists:
    name_series = prism_table[cell_line_name_col].astype(str).str.lower() if cell_line_name_col else pd.Series([], dtype=str)
    mock_terms = ["example", "mock", "toy", "dummy"]
    mock_name_count = name_series.str.contains("|".join(mock_terms), na=False).sum() if cell_line_name_col else 0

    has_too_few_rows = len(prism_table) < 5
    has_identical_lfc = False
    if "prism_doxorubicin_lfc" in prism_table.columns:
        has_identical_lfc = prism_table["prism_doxorubicin_lfc"].dropna().nunique() <= 1

    validation_rows.extend([
        {"check": "mock_cell_line_names", "status": "flag" if mock_name_count > 0 else "pass", "detail": int(mock_name_count)},
        {"check": "at_least_5_rows", "status": "flag" if has_too_few_rows else "pass", "detail": len(prism_table)},
        {"check": "lfc_values_not_all_identical", "status": "flag" if has_identical_lfc else "pass", "detail": str(has_identical_lfc)},
    ])

    print("Mock-name matches:", mock_name_count)
    print("Fewer than 5 rows:", has_too_few_rows)
    print("All PRISM doxorubicin LFC values identical:", has_identical_lfc)
else:
    print("Skipping mock-data checks until the processed table exists.")

In [ ]:
if processed_table_exists:
    required_response_cols = [
        "prism_doxorubicin_lfc",
        "relative_abundance_percent",
        "doxorubicin_sensitivity_score",
    ]
    missing_response_cols = [col for col in required_response_cols if col not in prism_table.columns]

    if not missing_response_cols:
        response_check = prism_table[required_response_cols].dropna().copy()
        response_check["expected_relative_abundance_percent"] = 100 * (2 ** response_check["prism_doxorubicin_lfc"])
        response_check["expected_sensitivity_score"] = -1 * response_check["prism_doxorubicin_lfc"]
        response_check["relative_abundance_abs_error"] = (
            response_check["relative_abundance_percent"] - response_check["expected_relative_abundance_percent"]
        ).abs()
        response_check["sensitivity_abs_error"] = (
            response_check["doxorubicin_sensitivity_score"] - response_check["expected_sensitivity_score"]
        ).abs()

        response_rows_checked = len(response_check)
        relative_abundance_ok = response_rows_checked > 0 and (response_check["relative_abundance_abs_error"] < 1e-8).all()
        sensitivity_ok = response_rows_checked > 0 and (response_check["sensitivity_abs_error"] < 1e-8).all()

        print("Rows checked:", response_rows_checked)
        display(response_check.head())
    else:
        response_rows_checked = 0
        relative_abundance_ok = False
        sensitivity_ok = False
        print("Missing response columns:", missing_response_cols)

    validation_rows.extend([
        {"check": "required_response_columns", "status": "pass" if not missing_response_cols else "flag", "detail": ", ".join(missing_response_cols)},
        {"check": "formula_rows_checked", "status": "info", "detail": response_rows_checked},
        {"check": "relative_abundance_formula", "status": "pass" if relative_abundance_ok else "flag", "detail": str(relative_abundance_ok)},
        {"check": "sensitivity_score_formula", "status": "pass" if sensitivity_ok else "flag", "detail": str(sensitivity_ok)},
    ])
else:
    print("Skipping PRISM LFC formula checks until the processed table exists.")

### Directionality

More negative PRISM LFC means lower relative abundance versus control after doxorubicin treatment, which means greater apparent sensitivity.

`doxorubicin_sensitivity_score` is defined as `-1 * prism_doxorubicin_lfc`, so higher sensitivity scores mean greater sensitivity by construction.

In [ ]:
if processed_table_exists and "doxorubicin_sensitivity_score" in prism_table.columns:
    display_cols = [col for col in [id_col, cell_line_name_col, "prism_doxorubicin_lfc", "relative_abundance_percent", "doxorubicin_sensitivity_score"] if col in prism_table.columns]

    most_sensitive = prism_table.sort_values("doxorubicin_sensitivity_score", ascending=False).head(5)
    least_sensitive = prism_table.sort_values("doxorubicin_sensitivity_score", ascending=True).head(5)

    print("Five most sensitive cell lines:")
    display(most_sensitive[display_cols])

    print("Five least sensitive cell lines:")
    display(least_sensitive[display_cols])
else:
    print("Skipping directionality tables until the processed table and sensitivity score exist.")

In [ ]:
validation_summary = pd.DataFrame(validation_rows).drop_duplicates(subset="check", keep="last")
validation_summary.to_csv(validation_summary_path, index=False)

display(validation_summary)
print("Saved validation summary to:", validation_summary_path)

## Validation Conclusion

At this checkpoint, the processed table is ready for Q2/Q3/Q5 only if:

- `data/processed/brca_prism_doxorubicin_modelling_table.csv` exists;
- it has real breast cancer cell-line names rather than mock/example names;
- it contains `prism_doxorubicin_lfc`, `relative_abundance_percent`, and `doxorubicin_sensitivity_score`;
- the response transformations match the formulas above;
- a useful subset of p53/DNA-damage expression columns is present.

If the processed table is missing, the blocker is data availability: place the raw DepMap/CCLE and PRISM files under `data/raw/depmap_data/`, rerun the preparation cells above, and then rerun this validation checkpoint.

Assumptions to carry into the report: PRISM LFC is a relative abundance or viability outcome versus control, not a gene-expression change and not a percentage of cells destroyed. The sensitivity score is a convenience transformation so that larger values mean greater doxorubicin sensitivity.

## TCGA-BRCA Patient Survival and Expression Preparation

This section prepares the patient-side table needed for Q1 and Q3.

The planned TCGA-BRCA table links tumour expression to clinical follow-up:

```text
TCGA-BRCA tumour expression + clinical survival follow-up -> patient modelling table
```

Unlike PRISM, TCGA-BRCA does not directly measure doxorubicin response. It is used here for a prognostic survival analysis: later notebooks can test whether p53/ODE scores or cell-line-derived expression signatures are associated with patient outcome.

Expected raw TCGA-BRCA files should be placed under `data/raw/tcga_brca/`. If those files are not available, the notebook reports the missing inputs and stops before creating any processed TCGA table.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

tcga_raw_dir = project_root / "data/raw/tcga_brca"
tcga_processed_path = project_root / "data/processed/tcga_brca_survival_expression_table.csv"


if tcga_raw_dir.exists():
    tcga_files = sorted([
        path for path in tcga_raw_dir.rglob("*")
        if path.is_file() and path.name not in [".DS_Store", ".gitkeep"]
    ])
else:
    tcga_files = []

print("Expected TCGA-BRCA raw directory:", tcga_raw_dir)
print("Directory exists:", tcga_raw_dir.exists())
print("Number of raw files found:", len(tcga_files))

if tcga_files:
    file_overview = pd.DataFrame({
        "file": [str(path.relative_to(project_root)) for path in tcga_files],
        "size_mb": [round(path.stat().st_size / 1024 / 1024, 2) for path in tcga_files],
    })
    display(file_overview)
else:
    print("No TCGA-BRCA raw files found. Place expression and clinical/survival files under data/raw/tcga_brca/.")

### Expected TCGA-BRCA Inputs

The notebook expects one expression file and one clinical or survival file.

Common acceptable file names include words such as:

- expression file: `expression`, `expr`, `tpm`, `fpkm`, `rsem`, `counts`, `transcriptome`, or `gene`;
- clinical file: `clinical`, `survival`, `follow`, or `phenotype`.

The exact TCGA export format can vary. The code below handles two simple expression layouts:

- one row per sample with gene symbols as columns;
- one row per gene with TCGA samples as columns.

If the files use a different structure, the notebook will need a small, visible adjustment after the real export files are available.

In [ ]:
expression_name_terms = ["expression", "expr", "tpm", "fpkm", "rsem", "counts", "transcriptome", "gene"]
preferred_expression_terms = ["expression", "expr", "tpm", "fpkm", "rsem", "counts", "transcriptome"]
lower_priority_expression_terms = ["annotation", "metadata", "sample", "clinical", "survival", "phenotype"]
clinical_name_terms = ["clinical", "survival", "follow", "phenotype"]

expression_candidates = [
    path for path in tcga_files
    if any(term in path.name.lower() for term in expression_name_terms)
]
expression_candidates = sorted(
    expression_candidates,
    key=lambda path: (
        -sum(term in path.name.lower() for term in preferred_expression_terms),
        sum(term in path.name.lower() for term in lower_priority_expression_terms),
        path.name.lower(),
    ),
)
clinical_candidates = [
    path for path in tcga_files
    if any(term in path.name.lower() for term in clinical_name_terms)
]

print("Expression file candidates:")
for path in expression_candidates:
    print("-", path.relative_to(project_root))

print("Clinical/survival file candidates:")
for path in clinical_candidates:
    print("-", path.relative_to(project_root))

missing_tcga_inputs = []
if not expression_candidates:
    missing_tcga_inputs.append("expression file")
if not clinical_candidates:
    missing_tcga_inputs.append("clinical/survival file")

if missing_tcga_inputs:
    print("Missing TCGA-BRCA input(s):", ", ".join(missing_tcga_inputs))
    print("Detailed TCGA table creation is skipped until those files are placed under data/raw/tcga_brca/.")

### Load TCGA-BRCA Expression

The p53/DNA-damage gene set is kept aligned with the DepMap/PRISM table:

`ATM, CHEK2, HIPK2, MDM2, PPM1D, SIAH1, TP53, WSB1, CDKN1A, BAX, BBC3, GADD45A, MDM4, ATR, CHEK1, CASP3`.

If several tumour samples are available for the same patient, the notebook keeps one sample per patient. Primary tumour samples with TCGA sample type code `01` are preferred; otherwise the first available sample is kept.

In [ ]:
p53_ddr_genes = [
    "ATM", "CHEK2", "HIPK2", "MDM2", "PPM1D", "SIAH1", "TP53", "WSB1",
    "CDKN1A", "BAX", "BBC3", "GADD45A", "MDM4", "ATR", "CHEK1", "CASP3",
]

expression_patient = pd.DataFrame()
tcga_found_genes = []
tcga_missing_genes = p53_ddr_genes.copy()

if expression_candidates:
    expression_table = pd.DataFrame()
    selected_expression_path = None

    for expression_path in expression_candidates:
        expression_sep = "	" if expression_path.suffix.lower() in [".tsv", ".txt"] else ","
        expression_raw = pd.read_csv(expression_path, sep=expression_sep, low_memory=False)

        genes_as_columns = [gene for gene in p53_ddr_genes if gene in expression_raw.columns]
        gene_column_candidates = [
            col for col in expression_raw.columns
            if col.lower() in ["gene", "gene_name", "gene_symbol", "hugo_symbol", "external_gene_name"]
        ]
        sample_column_candidates = [
            col for col in expression_raw.columns
            if col.lower() in ["sample", "sample_id", "sample_submitter_id", "barcode", "tcga_barcode"]
        ]

        if genes_as_columns:
            sample_id_col = sample_column_candidates[0] if sample_column_candidates else expression_raw.columns[0]
            expression_table = expression_raw[[sample_id_col] + genes_as_columns].copy()
            expression_table = expression_table.rename(columns={sample_id_col: "sample_id"})
            tcga_found_genes = genes_as_columns
            selected_expression_path = expression_path
            break

        if gene_column_candidates:
            gene_col = gene_column_candidates[0]
            expression_raw["gene_symbol_for_matching"] = expression_raw[gene_col].astype(str).str.split("|", regex=False).str[0]
            gene_rows = expression_raw[expression_raw["gene_symbol_for_matching"].isin(p53_ddr_genes)].copy()

            if not gene_rows.empty:
                annotation_cols = set(gene_column_candidates + ["gene_symbol_for_matching", "gene_id", "ensembl_gene_id", "description"])
                sample_cols = [col for col in gene_rows.columns if col not in annotation_cols]

                expression_table = gene_rows[["gene_symbol_for_matching"] + sample_cols].set_index("gene_symbol_for_matching").T.reset_index()
                expression_table = expression_table.rename(columns={"index": "sample_id"})
                tcga_found_genes = [gene for gene in p53_ddr_genes if gene in expression_table.columns]
                selected_expression_path = expression_path
                break

        print("Skipping expression candidate without requested genes:", expression_path.relative_to(project_root))

    if selected_expression_path:
        print("Using expression file:", selected_expression_path.relative_to(project_root))
        print("Expression table shape before patient filtering:", expression_table.shape)

    if not expression_table.empty and tcga_found_genes:
        expression_table["sample_id"] = expression_table["sample_id"].astype(str)
        expression_table["patient_id"] = expression_table["sample_id"].str[:12]
        expression_table["sample_type_code"] = expression_table["sample_id"].str[13:15]
        expression_table["is_primary_tumour"] = expression_table["sample_type_code"].eq("01")

        expression_table = expression_table.sort_values(["patient_id", "is_primary_tumour"], ascending=[True, False])
        expression_patient = expression_table.drop_duplicates("patient_id", keep="first")
        expression_patient = expression_patient[["patient_id", "sample_id"] + tcga_found_genes].copy()

        tcga_missing_genes = [gene for gene in p53_ddr_genes if gene not in tcga_found_genes]

        print("Patients with expression data:", len(expression_patient))
        print("Genes found:", tcga_found_genes)
        print("Genes missing:", tcga_missing_genes)
        display(expression_patient.head())
else:
    print("Skipping expression loading because no TCGA-BRCA expression file was found.")

### Load TCGA-BRCA Clinical and Survival Data

Overall survival is prepared using a simple rule:

- `os_event = 1` for patients with death recorded;
- `os_event = 0` for censored patients still alive at last follow-up;
- `os_time_days` uses `days_to_death` when available, otherwise `days_to_last_follow_up`.

Age and stage are kept if available because they are useful clinical covariates for later Cox regression.

In [ ]:
clinical_survival = pd.DataFrame()
selected_clinical_path = None

if clinical_candidates:
    patient_id_candidates = [
        "patient_id", "bcr_patient_barcode", "case_submitter_id", "submitter_id",
        "case_id", "PATIENT_ID", "Patient ID",
    ]

    for clinical_path in clinical_candidates:
        clinical_sep = "," if clinical_path.suffix.lower() == ".csv" else "	"
        clinical_raw = pd.read_csv(clinical_path, sep=clinical_sep, low_memory=False)

        patient_id_col = next((col for col in patient_id_candidates if col in clinical_raw.columns), None)
        if patient_id_col is None:
            print("Skipping clinical candidate without patient ID:", clinical_path.relative_to(project_root))
            continue

        candidate_survival = pd.DataFrame()
        candidate_survival["patient_id"] = clinical_raw[patient_id_col].astype(str).str[:12]

        days_to_death_col = next((col for col in ["days_to_death", "DAYS_TO_DEATH"] if col in clinical_raw.columns), None)
        days_to_follow_up_col = next((col for col in ["days_to_last_follow_up", "days_to_last_followup", "DAYS_TO_LAST_FOLLOWUP"] if col in clinical_raw.columns), None)
        vital_status_col = next((col for col in ["vital_status", "VITAL_STATUS", "OS_STATUS", "os_status"] if col in clinical_raw.columns), None)
        os_time_col = next((col for col in ["OS.time", "OS_MONTHS", "overall_survival", "overall_survival_months"] if col in clinical_raw.columns), None)
        os_event_col = next((col for col in ["OS", "os", "OS_EVENT", "os_event", "event", "EVENT", "overall_survival_event"] if col in clinical_raw.columns), None)

        if days_to_death_col:
            days_to_death = pd.to_numeric(clinical_raw[days_to_death_col], errors="coerce")
        else:
            days_to_death = pd.Series([pd.NA] * len(clinical_raw))

        if days_to_follow_up_col:
            days_to_follow_up = pd.to_numeric(clinical_raw[days_to_follow_up_col], errors="coerce")
        else:
            days_to_follow_up = pd.Series([pd.NA] * len(clinical_raw))

        day_based_time = days_to_death.fillna(days_to_follow_up)
        if day_based_time.notna().any():
            candidate_survival["os_time_days"] = day_based_time
        elif os_time_col:
            os_time = pd.to_numeric(clinical_raw[os_time_col], errors="coerce")
            if "month" in os_time_col.lower():
                os_time = os_time * 30.44
            candidate_survival["os_time_days"] = os_time
        else:
            print("Skipping clinical candidate without survival time:", clinical_path.relative_to(project_root))
            continue

        if vital_status_col:
            status_text = clinical_raw[vital_status_col].astype(str).str.lower()
            event_from_status = status_text.str.contains("dead|deceased", regex=True) | status_text.str.startswith("1")
            candidate_survival["os_event"] = (days_to_death.notna() | event_from_status).astype(int)
        elif os_event_col:
            event_numeric = pd.to_numeric(clinical_raw[os_event_col], errors="coerce")
            if event_numeric.notna().any():
                candidate_survival["os_event"] = event_numeric
            else:
                event_text = clinical_raw[os_event_col].astype(str).str.lower()
                candidate_survival["os_event"] = (
                    event_text.str.contains("dead|deceased", regex=True) | event_text.str.startswith("1")
                ).astype(int)
        else:
            candidate_survival["os_event"] = days_to_death.notna().astype(int)

        age_col = next((col for col in ["age_at_index", "age_at_diagnosis", "age", "AGE"] if col in clinical_raw.columns), None)
        if age_col:
            age_values = pd.to_numeric(clinical_raw[age_col], errors="coerce")
            if age_values.dropna().median() > 120:
                age_values = age_values / 365.25
            candidate_survival["age_years"] = age_values

        stage_col = next((col for col in ["ajcc_pathologic_stage", "pathologic_stage", "tumor_stage", "stage", "STAGE"] if col in clinical_raw.columns), None)
        if stage_col:
            candidate_survival["stage"] = clinical_raw[stage_col]

        candidate_survival = candidate_survival.drop_duplicates("patient_id", keep="first")
        candidate_survival = candidate_survival[candidate_survival["patient_id"].notna()].copy()
        usable_rows = candidate_survival["os_time_days"].notna() & candidate_survival["os_event"].notna()

        if usable_rows.any():
            clinical_survival = candidate_survival
            selected_clinical_path = clinical_path
            break

        print("Skipping clinical candidate without usable survival rows:", clinical_path.relative_to(project_root))

    if selected_clinical_path:
        print("Using clinical/survival file:", selected_clinical_path.relative_to(project_root))
        print("Patients with clinical rows:", len(clinical_survival))
        print("Patients with usable survival time:", clinical_survival["os_time_days"].notna().sum())
        print("Patients with usable survival event:", clinical_survival["os_event"].notna().sum())
        print("Events:", clinical_survival["os_event"].fillna(0).sum())
        display(clinical_survival.head())
else:
    print("Skipping clinical loading because no TCGA-BRCA clinical/survival file was found.")

### Join Expression and Survival Data

This generic check previews whether expression and survival data can be joined at patient level.

It does not save final TCGA output files. The UCSC Xena and validation sections below write:

`data/processed/tcga_brca_survival_expression_table.csv`

`tables/tcga_brca_data_validation_summary.csv`

`figures/tcga_brca_survival_time_distribution.png`

In [ ]:
tcga_summary_rows = [
    {"metric": "raw_tcga_directory_exists", "value": tcga_raw_dir.exists()},
    {"metric": "raw_file_count", "value": len(tcga_files)},
    {"metric": "expression_candidate_count", "value": len(expression_candidates)},
    {"metric": "clinical_candidate_count", "value": len(clinical_candidates)},
    {"metric": "p53_ddr_gene_count_requested", "value": len(p53_ddr_genes)},
    {"metric": "p53_ddr_gene_count_found", "value": len(tcga_found_genes)},
    {"metric": "p53_ddr_genes_found", "value": ", ".join(tcga_found_genes)},
    {"metric": "p53_ddr_genes_missing", "value": ", ".join(tcga_missing_genes)},
]

if not expression_patient.empty and not clinical_survival.empty:
    tcga_patient_table = clinical_survival.merge(expression_patient, on="patient_id", how="inner")
    tcga_patient_table = tcga_patient_table[tcga_patient_table["os_time_days"].notna()].copy()
    tcga_patient_table = tcga_patient_table[tcga_patient_table[tcga_found_genes].notna().any(axis=1)].copy()

    tcga_summary_rows.extend([
        {"metric": "patients_with_expression_data", "value": len(expression_patient)},
        {"metric": "patients_with_clinical_data", "value": len(clinical_survival)},
        {"metric": "patients_with_usable_survival_data", "value": clinical_survival["os_time_days"].notna().sum()},
        {"metric": "patients_in_joined_table", "value": len(tcga_patient_table)},
        {"metric": "survival_events", "value": int(tcga_patient_table["os_event"].fillna(0).sum())},
        {"metric": "survival_censored", "value": int((tcga_patient_table["os_event"].fillna(0) == 0).sum())},
    ])

    tcga_summary = pd.DataFrame(tcga_summary_rows)

    plt.figure(figsize=(7, 4))
    plt.hist(tcga_patient_table["os_time_days"].dropna(), bins=30, edgecolor="black")
    plt.xlabel("Overall survival time (days)")
    plt.ylabel("Number of patients")
    plt.title("TCGA-BRCA survival time distribution")
    plt.tight_layout()
    plt.show()

    print("Prepared a generic TCGA patient preview table. The Xena section below writes the final processed table to:", tcga_processed_path)
    print("This preview did not save a CSV table or figure.")
    display(tcga_summary)
    display(tcga_patient_table.head())
else:
    tcga_summary = pd.DataFrame(tcga_summary_rows)
    display(tcga_summary)

    skip_reasons = []
    if not expression_candidates:
        skip_reasons.append("no expression file candidates were found")
    elif expression_patient.empty:
        skip_reasons.append("expression candidates were found, but no usable p53/DNA-damage expression table was loaded")

    if not clinical_candidates:
        skip_reasons.append("no clinical/survival file candidates were found")
    elif clinical_survival.empty:
        skip_reasons.append("clinical candidates were found, but no usable survival table was loaded")

    print("TCGA processed outputs were not created:", "; ".join(skip_reasons))

### TCGA-BRCA Preparation Interpretation

When the raw TCGA-BRCA files are available, this section should report:

- number of TCGA-BRCA patients with expression data;
- number with usable overall survival data;
- number in the joined patient modelling table;
- number of p53/DNA-damage genes found;
- which genes overlap with the DepMap/PRISM table.

Current limitation: no TCGA-BRCA raw expression or clinical files are present under `data/raw/tcga_brca/` in this working tree, so no patient modelling table has been generated yet.

Assumption to carry forward: TCGA-BRCA supports prognostic survival testing. It should not be described as a direct doxorubicin-response dataset.

## Create TCGA-BRCA Survival Expression Table from UCSC Xena

The previous preparation cells define the expected TCGA-BRCA workflow. This section now creates the actual processed patient-level table using simple UCSC Xena files.

Expected raw files under `data/raw/tcga_brca/`:

- `HiSeqV2.gz`: TCGA-BRCA gene expression matrix;
- `BRCA_survival.txt`: survival endpoints from the Xena `survival/BRCA_survival.txt` dataset;
- `BRCA_clinicalMatrix`: clinical phenotype matrix.

If any file is missing, the notebook tries to download it from UCSC Xena. If downloading fails, it prints the exact file and URL that should be downloaded manually. No mock data are created.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd


project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw_tcga_dir = project_root / "data/raw/tcga_brca"
processed_tcga_path = project_root / "data/processed/tcga_brca_survival_expression_table.csv"
raw_tcga_dir.mkdir(parents=True, exist_ok=True)
processed_tcga_path.parent.mkdir(parents=True, exist_ok=True)

xena_files = {
    "expression": {
        "path": raw_tcga_dir / "HiSeqV2.gz",
        "url": "https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/HiSeqV2.gz",
    },
    "survival": {
        "path": raw_tcga_dir / "BRCA_survival.txt",
        "url": "https://tcga.xenahubs.net/download/survival/BRCA_survival.txt",
    },
    "clinical": {
        "path": raw_tcga_dir / "BRCA_clinicalMatrix",
        "url": "https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/BRCA_clinicalMatrix",
    },
}

for label, file_info in xena_files.items():
    path = file_info["path"]
    url = file_info["url"]

    if path.exists():
        print(f"Found {label} file:", path.relative_to(project_root))
    else:
        print(f"Missing {label} file. Trying download from UCSC Xena:", url)
        try:
            urlretrieve(url, path)
            print("Downloaded:", path.relative_to(project_root))
        except Exception as error:
            print("Download failed:", error)
            print("Please manually place this file at:", path)
            print("Source URL:", url)

raw_xena_files_available = all(file_info["path"].exists() for file_info in xena_files.values())
print("All required TCGA-BRCA raw files available:", raw_xena_files_available)

### Load Xena Expression and Keep p53/DNA-Damage Genes

`HiSeqV2.gz` is a gene-by-sample matrix. We keep only the pre-specified p53/DNA-damage genes and transpose the table so each row is a TCGA sample.

TCGA sample barcodes are converted to patient IDs using the first 12 characters. If multiple samples map to the same patient, primary tumour samples with sample type code `01` are preferred, then the first remaining sample is kept.

In [ ]:
p53_ddr_genes = [
    "ATM", "CHEK2", "HIPK2", "MDM2", "PPM1D", "SIAH1", "TP53", "WSB1",
    "CDKN1A", "BAX", "BBC3", "GADD45A", "MDM4", "ATR", "CHEK1", "CASP3",
]

xena_expression_patient = pd.DataFrame()
xena_found_genes = []
xena_missing_genes = p53_ddr_genes.copy()

if raw_xena_files_available:
    expression_raw = pd.read_csv(xena_files["expression"]["path"], sep="	", index_col=0)
    expression_raw.index = expression_raw.index.astype(str)

    xena_found_genes = [gene for gene in p53_ddr_genes if gene in expression_raw.index]
    xena_missing_genes = [gene for gene in p53_ddr_genes if gene not in expression_raw.index]

    expression_selected = expression_raw.loc[xena_found_genes].T.reset_index()
    expression_selected = expression_selected.rename(columns={"index": "sample_id"})
    expression_selected["sample_id"] = expression_selected["sample_id"].astype(str)
    expression_selected["patient_id"] = expression_selected["sample_id"].str[:12]
    expression_selected["sample_type_code"] = expression_selected["sample_id"].str[13:15]
    expression_selected["is_primary_tumour"] = expression_selected["sample_type_code"].eq("01")

    expression_selected = expression_selected.sort_values(["patient_id", "is_primary_tumour"], ascending=[True, False])
    xena_expression_patient = expression_selected.drop_duplicates("patient_id", keep="first")
    xena_expression_patient = xena_expression_patient[["patient_id", "sample_id"] + xena_found_genes].copy()

    print("Expression samples loaded:", len(expression_selected))
    print("Patients with one selected expression sample:", len(xena_expression_patient))
    print("Genes found:", xena_found_genes)
    print("Genes missing:", xena_missing_genes)
    display(xena_expression_patient.head())
else:
    print("Skipping expression loading because the required TCGA-BRCA raw files are not all available.")

### Load Xena Survival and Clinical Covariates

The Xena survival file already contains overall survival fields:

- `OS.time`: overall survival time in days;
- `OS`: event indicator, where 1 means death/event and 0 means censored/alive.

The clinical matrix is used only for simple covariates: age and stage, if available.

In [ ]:
xena_survival_patient = pd.DataFrame()
xena_clinical_patient = pd.DataFrame()

if raw_xena_files_available:
    survival_raw = pd.read_csv(xena_files["survival"]["path"], sep="	")
    survival_raw["patient_id"] = survival_raw["_PATIENT"].astype(str).str[:12]
    xena_survival_patient = survival_raw[["patient_id", "OS.time", "OS"]].copy()
    xena_survival_patient = xena_survival_patient.rename(columns={
        "OS.time": "overall_survival_time",
        "OS": "overall_survival_event",
    })
    xena_survival_patient["overall_survival_time"] = pd.to_numeric(xena_survival_patient["overall_survival_time"], errors="coerce")
    xena_survival_patient["overall_survival_event"] = pd.to_numeric(xena_survival_patient["overall_survival_event"], errors="coerce")
    xena_survival_patient = xena_survival_patient.drop_duplicates("patient_id", keep="first")

    clinical_raw = pd.read_csv(xena_files["clinical"]["path"], sep="	", low_memory=False)
    clinical_raw["patient_id"] = clinical_raw["_PATIENT"].astype(str).str[:12]

    xena_clinical_patient = clinical_raw[["patient_id"]].copy()

    age_col = None
    for col in ["age_at_initial_pathologic_diagnosis", "Age_at_Initial_Pathologic_Diagnosis_nature2012"]:
        if col in clinical_raw.columns:
            age_col = col
            break
    if age_col:
        xena_clinical_patient["age"] = pd.to_numeric(clinical_raw[age_col], errors="coerce")

    stage_col = None
    for col in ["pathologic_stage", "AJCC_Stage_nature2012", "Converted_Stage_nature2012"]:
        if col in clinical_raw.columns:
            stage_col = col
            break
    if stage_col:
        xena_clinical_patient["stage"] = clinical_raw[stage_col]

    xena_clinical_patient = xena_clinical_patient.drop_duplicates("patient_id", keep="first")

    print("Patients with survival data:", len(xena_survival_patient))
    print("Survival events/deaths:", int((xena_survival_patient["overall_survival_event"] == 1).sum()))
    print("Censored patients:", int((xena_survival_patient["overall_survival_event"] == 0).sum()))
    print("Clinical covariates kept:", [col for col in xena_clinical_patient.columns if col != "patient_id"])
    display(xena_survival_patient.head())
    display(xena_clinical_patient.head())
else:
    print("Skipping survival and clinical loading because the required TCGA-BRCA raw files are not all available.")

### Join and Save the TCGA-BRCA Patient Table

The output contains one row per patient and is saved to:

`data/processed/tcga_brca_survival_expression_table.csv`

This table supports later Q1 and Q3 analyses. It does not measure doxorubicin response directly; it supports prognostic survival testing in TCGA-BRCA.

In [ ]:
xena_tcga_table = pd.DataFrame()

if raw_xena_files_available and not xena_expression_patient.empty and not xena_survival_patient.empty:
    xena_tcga_table = xena_survival_patient.merge(xena_expression_patient, on="patient_id", how="inner")

    if not xena_clinical_patient.empty:
        xena_tcga_table = xena_tcga_table.merge(xena_clinical_patient, on="patient_id", how="left")

    xena_tcga_table = xena_tcga_table[xena_tcga_table["overall_survival_time"].notna()].copy()
    xena_tcga_table = xena_tcga_table[xena_tcga_table["overall_survival_event"].isin([0, 1])].copy()
    xena_tcga_table = xena_tcga_table[xena_tcga_table[xena_found_genes].notna().any(axis=1)].copy()

    first_columns = ["patient_id", "sample_id", "overall_survival_time", "overall_survival_event"]
    clinical_columns = [col for col in ["age", "stage"] if col in xena_tcga_table.columns]
    xena_tcga_table = xena_tcga_table[first_columns + clinical_columns + xena_found_genes]

    xena_tcga_table.to_csv(processed_tcga_path, index=False)

    print("Saved TCGA-BRCA survival expression table to:", processed_tcga_path.relative_to(project_root))
    print("Rows/patients:", len(xena_tcga_table))
    print("Columns:", len(xena_tcga_table.columns))
    print("Events/deaths:", int((xena_tcga_table["overall_survival_event"] == 1).sum()))
    print("Censored:", int((xena_tcga_table["overall_survival_event"] == 0).sum()))
    print("p53/DNA-damage gene columns:", xena_found_genes)
    display(xena_tcga_table.head())
else:
    print("Processed TCGA table was not created because required raw inputs or parsed tables are missing.")

### TCGA-BRCA Table Creation Interpretation

The processed TCGA-BRCA table is intended for prognostic survival analysis. TCGA-BRCA provides tumour expression and clinical follow-up, but it does not directly measure doxorubicin response.

This table will support Q1 by linking p53/DNA-damage expression or mechanistic scores to patient survival, and Q3 by allowing later comparison with cell-line-derived signatures. Survival or machine-learning models are not run in this notebook.

## TCGA-BRCA Data Validation Checkpoint

This lightweight checkpoint validates the processed TCGA-BRCA survival and expression table before Q1 or Q3 survival models are run.

It checks that the table has one row per patient, usable overall survival fields, clinical covariates where available, and expression columns for the p53/DNA-damage genes used throughout the assignment.

No Cox regression, Kaplan-Meier modelling, PRISM signature transfer, p53 ODE scoring, GDSC analysis, or LINCS analysis is run here.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

tcga_table_path = project_root / "data/processed/tcga_brca_survival_expression_table.csv"
prism_table_path = project_root / "data/processed/brca_prism_doxorubicin_modelling_table.csv"
tcga_validation_summary_path = project_root / "tables/tcga_brca_data_validation_summary.csv"
tcga_validation_note_path = project_root / "notes/tcga_brca_data_validation_summary.md"
tcga_survival_figure_path = project_root / "figures/tcga_brca_survival_time_distribution.png"

for folder in [tcga_validation_summary_path.parent, tcga_validation_note_path.parent, tcga_survival_figure_path.parent]:
    folder.mkdir(exist_ok=True)

p53_ddr_genes = [
    "ATM", "CHEK2", "HIPK2", "MDM2", "PPM1D", "SIAH1", "TP53", "WSB1",
    "CDKN1A", "BAX", "BBC3", "GADD45A", "MDM4", "ATR", "CHEK1", "CASP3",
]

validation_rows = []
tcga_table_exists = tcga_table_path.exists()

print("Processed TCGA-BRCA table:", tcga_table_path)
print("Exists:", tcga_table_exists)

validation_rows.append({
    "check": "processed_tcga_table_exists",
    "status": "pass" if tcga_table_exists else "blocker",
    "detail": str(tcga_table_path.relative_to(project_root)),
})

In [ ]:
tcga_table = pd.DataFrame()
patient_id_col = None
survival_time_col = None
survival_event_col = None
clinical_covariates_present = []
tcga_gene_cols = []
tcga_missing_genes = p53_ddr_genes.copy()

if tcga_table_exists:
    tcga_table = pd.read_csv(tcga_table_path)

    patient_id_candidates = ["patient_id", "Patient ID", "bcr_patient_barcode", "case_submitter_id"]
    survival_time_candidates = ["overall_survival_time", "os_time_days", "survival_time_days", "overall_survival_days", "OS.time", "OS_MONTHS"]
    survival_event_candidates = ["os_event", "survival_event", "overall_survival_event", "OS", "OS_EVENT", "event"]

    patient_id_col = next((col for col in patient_id_candidates if col in tcga_table.columns), None)
    survival_time_col = next((col for col in survival_time_candidates if col in tcga_table.columns), None)
    survival_event_col = next((col for col in survival_event_candidates if col in tcga_table.columns), None)

    clinical_covariates_present = [
        col for col in tcga_table.columns
        if col.lower() in ["age", "age_years", "age_at_diagnosis", "age_at_index", "stage", "pathologic_stage", "ajcc_pathologic_stage"]
    ]
    tcga_gene_cols = [gene for gene in p53_ddr_genes if gene in tcga_table.columns]
    tcga_missing_genes = [gene for gene in p53_ddr_genes if gene not in tcga_table.columns]

    print("Rows:", len(tcga_table))
    print("Columns:", len(tcga_table.columns))
    print("Patient ID column:", patient_id_col)
    print("Survival time column:", survival_time_col)
    print("Survival event column:", survival_event_col)
    print("Clinical covariates present:", clinical_covariates_present)
    print("p53/DNA-damage genes present:", tcga_gene_cols)
    print("p53/DNA-damage genes missing:", tcga_missing_genes)
    display(tcga_table.head())

    validation_rows.extend([
        {"check": "row_count", "status": "info", "detail": len(tcga_table)},
        {"check": "column_count", "status": "info", "detail": len(tcga_table.columns)},
        {"check": "patient_id_column", "status": "pass" if patient_id_col else "flag", "detail": patient_id_col or "not found"},
        {"check": "survival_time_column", "status": "pass" if survival_time_col else "flag", "detail": survival_time_col or "not found"},
        {"check": "survival_event_column", "status": "pass" if survival_event_col else "flag", "detail": survival_event_col or "not found"},
        {"check": "clinical_covariates_present", "status": "info", "detail": ", ".join(clinical_covariates_present)},
        {"check": "p53_ddr_genes_present", "status": "info", "detail": ", ".join(tcga_gene_cols)},
        {"check": "p53_ddr_genes_missing", "status": "info", "detail": ", ".join(tcga_missing_genes)},
    ])
else:
    print("Cannot run detailed TCGA validation because the processed table is missing.")
    print("Expected file: data/processed/tcga_brca_survival_expression_table.csv")

In [ ]:
survival_time_numeric = False
survival_time_non_negative = False
survival_event_observed = False
survival_event_binary = False
survival_time_unit = "not computed"
event_count = None
censored_count = None
missing_survival_time_count = None
missing_survival_event_count = None

if tcga_table_exists and survival_time_col and survival_event_col:
    survival_time = pd.to_numeric(tcga_table[survival_time_col], errors="coerce")
    if "month" in survival_time_col.lower():
        survival_time = survival_time * 30.44
        survival_time_unit = "months converted to days"
    else:
        survival_time_unit = "days"

    survival_event = pd.to_numeric(tcga_table[survival_event_col], errors="coerce")

    survival_time_numeric = survival_time.notna().sum() > 0
    survival_time_non_negative = survival_time_numeric and (survival_time.dropna() >= 0).all()
    survival_event_values = sorted(survival_event.dropna().unique().tolist())
    survival_event_observed = survival_event.notna().sum() > 0
    survival_event_binary = survival_event_observed and set(survival_event_values).issubset({0, 1})

    missing_survival_time_count = int(survival_time.isna().sum())
    missing_survival_event_count = int(survival_event.isna().sum())

    if survival_event_observed:
        event_count = int((survival_event == 1).sum())
        censored_count = int((survival_event == 0).sum())

    print("Survival time numeric:", survival_time_numeric)
    print("Survival time unit used for validation:", survival_time_unit)
    print("Survival time non-negative:", survival_time_non_negative)
    print("Survival event values:", survival_event_values)
    print("Survival event observed:", survival_event_observed)
    print("Deaths/events:", event_count if event_count is not None else "not computed")
    print("Censored patients:", censored_count if censored_count is not None else "not computed")
    print("Missing survival times:", missing_survival_time_count)
    print("Missing survival events:", missing_survival_event_count)

    plt.figure(figsize=(7, 4))
    plt.hist(survival_time.dropna(), bins=30, edgecolor="black")
    plt.xlabel("Overall survival time (days)")
    plt.ylabel("Number of patients")
    plt.title("TCGA-BRCA survival time distribution")
    plt.tight_layout()
    plt.savefig(tcga_survival_figure_path, dpi=200)
    plt.show()

    validation_rows.extend([
        {"check": "survival_time_numeric", "status": "pass" if survival_time_numeric else "flag", "detail": str(survival_time_numeric)},
        {"check": "survival_time_unit", "status": "info", "detail": survival_time_unit},
        {"check": "survival_time_non_negative", "status": "pass" if survival_time_non_negative else "flag", "detail": str(survival_time_non_negative)},
        {"check": "survival_event_observed", "status": "pass" if survival_event_observed else "flag", "detail": str(survival_event_observed)},
        {"check": "survival_event_binary", "status": "pass" if survival_event_binary else "flag", "detail": str(survival_event_values)},
        {"check": "event_count", "status": "info", "detail": event_count if event_count is not None else "not computed"},
        {"check": "censored_count", "status": "info", "detail": censored_count if censored_count is not None else "not computed"},
        {"check": "missing_survival_time_count", "status": "info", "detail": missing_survival_time_count},
        {"check": "missing_survival_event_count", "status": "info", "detail": missing_survival_event_count},
    ])
else:
    print("Skipping survival-field checks until the TCGA table and survival columns are available.")

In [ ]:
one_row_per_patient = False
duplicate_patient_count = None
data_level = "unknown"
patient_sample_rule = "TCGA sample IDs are converted to 12-character patient IDs; primary tumour sample type code 01 is preferred when multiple tumour samples exist."

if tcga_table_exists and patient_id_col:
    duplicate_patient_count = int(tcga_table[patient_id_col].duplicated().sum())
    one_row_per_patient = duplicate_patient_count == 0
    sample_id_present = "sample_id" in tcga_table.columns
    data_level = "patient-level" if one_row_per_patient else "sample-level or duplicated patient-level"

    print("One row per patient:", one_row_per_patient)
    print("Duplicate patient IDs:", duplicate_patient_count)
    print("Sample ID column present:", sample_id_present)
    print("Data level:", data_level)
    print("Patient/sample rule:", patient_sample_rule)

    validation_rows.extend([
        {"check": "one_row_per_patient", "status": "pass" if one_row_per_patient else "flag", "detail": str(one_row_per_patient)},
        {"check": "duplicate_patient_id_count", "status": "pass" if duplicate_patient_count == 0 else "flag", "detail": duplicate_patient_count},
        {"check": "data_level", "status": "info", "detail": data_level},
        {"check": "patient_sample_rule", "status": "info", "detail": patient_sample_rule},
    ])
else:
    print("Skipping patient/sample structure checks until the TCGA table and patient ID column are available.")

In [ ]:
prism_genes = []
genes_present_in_both = []
same_gene_set_for_transfer = False

if prism_table_path.exists():
    prism_table = pd.read_csv(prism_table_path, nrows=1)
    prism_genes = [gene for gene in p53_ddr_genes if gene in prism_table.columns]
else:
    print("PRISM/DepMap table is missing, so gene-set comparison is skipped:", prism_table_path)

if tcga_table_exists:
    genes_present_in_both = [gene for gene in p53_ddr_genes if gene in tcga_gene_cols and gene in prism_genes]
    same_gene_set_for_transfer = set(tcga_gene_cols) == set(prism_genes) and len(genes_present_in_both) > 0

    print("p53/DNA-damage genes in TCGA-BRCA:", tcga_gene_cols)
    print("p53/DNA-damage genes in PRISM/DepMap:", prism_genes)
    print("Genes present in both:", genes_present_in_both)
    print("Same gene set available for transfer/signature analysis:", same_gene_set_for_transfer)

validation_rows.extend([
    {"check": "prism_table_exists", "status": "pass" if prism_table_path.exists() else "flag", "detail": str(prism_table_path.relative_to(project_root))},
    {"check": "p53_ddr_genes_in_prism", "status": "info", "detail": ", ".join(prism_genes)},
    {"check": "p53_ddr_genes_in_both_tcga_and_prism", "status": "info", "detail": ", ".join(genes_present_in_both)},
    {"check": "same_gene_set_for_transfer", "status": "pass" if same_gene_set_for_transfer else "flag", "detail": str(same_gene_set_for_transfer)},
])

In [ ]:
ready_for_q1_q3 = (
    tcga_table_exists
    and patient_id_col is not None
    and survival_time_col is not None
    and survival_event_col is not None
    and survival_time_numeric
    and survival_time_non_negative
    and survival_event_observed
    and survival_event_binary
    and one_row_per_patient
    and len(tcga_gene_cols) > 0
    and len(genes_present_in_both) > 0
)

validation_rows.append({
    "check": "ready_for_q1_q3_survival_models",
    "status": "pass" if ready_for_q1_q3 else "blocker",
    "detail": str(ready_for_q1_q3),
})

validation_summary = pd.DataFrame(validation_rows).drop_duplicates(subset="check", keep="last")
validation_summary.to_csv(tcga_validation_summary_path, index=False)

display(validation_summary)
print("Saved TCGA validation summary to:", tcga_validation_summary_path)

if tcga_table_exists and survival_time_col and survival_event_col:
    figure_message = f"Saved survival-time distribution figure to: {tcga_survival_figure_path.relative_to(project_root)}"
else:
    figure_message = "Survival-time distribution figure was not created because the TCGA table or survival columns are unavailable."
print(figure_message)

### TCGA-BRCA Validation Interpretation

Use the validation summary below as a data quality checkpoint before Q1 and Q3 survival modelling.

The table is ready for Q1/Q3 only if it has one row per patient, usable survival time and event columns, interpretable clinical covariates where available, and overlapping p53/DNA-damage expression genes with the PRISM/DepMap table.

In [ ]:
if tcga_table_exists:
    usable_patients = len(tcga_table)
    age_available = any("age" in col.lower() for col in clinical_covariates_present)
    stage_available = any("stage" in col.lower() for col in clinical_covariates_present)
    blockers = validation_summary.loc[validation_summary["status"].isin(["blocker", "flag"]), "check"].tolist()
else:
    usable_patients = 0
    age_available = False
    stage_available = False
    blockers = ["processed_tcga_table_exists"]

event_count_text = event_count if event_count is not None else "not computed"
censored_count_text = censored_count if censored_count is not None else "not computed"

summary_lines = [
    "# TCGA-BRCA Data Validation Summary",
    "",
    f"Processed table present: {tcga_table_exists}",
    f"Usable TCGA-BRCA patients: {usable_patients}",
    f"Deaths/events: {event_count_text}",
    f"Censored patients: {censored_count_text}",
    f"Age available: {age_available}",
    f"Stage available: {stage_available}",
    f"p53/DNA-damage genes present in TCGA-BRCA: {', '.join(tcga_gene_cols) if tcga_gene_cols else 'none'}",
    f"p53/DNA-damage genes present in both TCGA-BRCA and PRISM/DepMap: {', '.join(genes_present_in_both) if genes_present_in_both else 'none'}",
    f"Ready for Q1/Q3 survival modelling: {ready_for_q1_q3}",
    f"Blockers or flags: {', '.join(blockers) if blockers else 'none'}",
    "",
    "Assumption: TCGA-BRCA supports prognostic survival analysis, not direct doxorubicin-response modelling.",
]

tcga_validation_note_path.write_text("\n".join(summary_lines) + "\n")
print("Saved markdown summary to:", tcga_validation_note_path)
print("\n".join(summary_lines))